# Notebook 3: Point-in-Time Investigation
Reconstructing the exact data context SupplyAgent operated on at 08:14
using Delta Time Travel, bi-temporal analysis, and OPTIMIZE.

**The question:** Why did the agent order 8000 gallons of oat milk on a rainy 45F morning?
**The answer:** It didn't hallucinate. It was data pipeline poisoning.


In [0]:
from pyspark.sql import functions as F


## Step 1: The Black Box Problem
The live table looks completely normal. An analyst seeing this right now
would have no explanation for the agent's decision.


In [0]:
print("Current live state of world_state (what anyone sees right now):")
spark.sql("""
    SELECT store, inventory_gallons, weather_temp_f,
           weather_condition, event_timestamp, source
    FROM workspace.supply_agent_demo.world_state
    WHERE store = 'Downtown'
    ORDER BY event_timestamp DESC
""").show()


## Step 2: The Suspicious Decision
The agent log shows an 08:14 order that makes no sense given the current data.


In [0]:
print("Agent memory log:")
spark.sql("""
    SELECT action_timestamp, decision, reasoning,
           context_inventory, context_temp_f
    FROM workspace.supply_agent_demo.agent_memory
    WHERE store = 'Downtown'
    ORDER BY action_timestamp
""").show(truncate=False)


## Step 3: Delta Time Travel - Reconstruct the Agent's Reality
Query world_state as it existed when the agent made its decision.
VERSION AS OF resolves to the corrupted_sensor commit.


In [0]:
corrupted_version = spark.sql("""
    DESCRIBE HISTORY workspace.supply_agent_demo.world_state
""").filter(
    F.col("userMetadata") == "corrupted_sensor"
).select("version").collect()[0]["version"]

print(f"Corrupted sensor commit is Delta version: {corrupted_version}")
print("World state as the agent saw it at 08:14:")

spark.sql(f"""
    SELECT store, inventory_gallons, weather_temp_f,
           weather_condition, event_timestamp, source
    FROM workspace.supply_agent_demo.world_state
    VERSION AS OF {corrupted_version}
    WHERE store = 'Downtown'
    ORDER BY event_timestamp DESC
    LIMIT 1
""").show()

# inventory_gallons = -999, weather_temp_f = 102 - the smoking gun


## Step 4: Bi-Temporal Audit
Two timelines exist for every data point:
- **event_timestamp**: when the sensor claimed the reading occurred
- **system_timestamp**: when the record was committed to Delta

Standard databases only preserve one. Delta preserves both.
The gap between them is where the audit trail lives.


In [0]:
print("Bi-temporal audit: what the agent saw vs. what the live table shows now:")
spark.sql("""
    SELECT
        agent_mem.action_timestamp                  AS agent_decision_time,
        agent_mem.context_inventory                 AS inventory_agent_saw,
        agent_mem.context_temp_f                    AS temp_agent_saw,
        live_state.inventory_gallons                AS inventory_live_now,
        live_state.weather_temp_f                   AS temp_live_now,
        agent_mem.system_timestamp                  AS decision_committed_to_delta,
        live_state.event_timestamp                  AS latest_sensor_event_time
    FROM workspace.supply_agent_demo.agent_memory     AS agent_mem
    JOIN workspace.supply_agent_demo.world_state      AS live_state
      ON agent_mem.store = live_state.store
    WHERE agent_mem.store = 'Downtown'
      AND agent_mem.action_timestamp = '2026-05-19 08:14:00'
    ORDER BY live_state.event_timestamp DESC
    LIMIT 1
""").show(truncate=False)


## Step 5: Commit Bloat - The Cost of High-Frequency Agent Writes
Every agent decision is a Delta commit. At production scale this creates
small-file bloat that degrades Time Travel query performance.
OPTIMIZE compacts the files. ZORDER improves query locality.


In [0]:
print("File count before OPTIMIZE:")
spark.sql("""
    DESCRIBE DETAIL workspace.supply_agent_demo.world_state
""").select("numFiles", "sizeInBytes").show()

spark.sql("""
    OPTIMIZE workspace.supply_agent_demo.world_state
    ZORDER BY (store, event_timestamp)
""")

print("File count after OPTIMIZE:")
spark.sql("""
    DESCRIBE DETAIL workspace.supply_agent_demo.world_state
""").select("numFiles", "sizeInBytes").show()


## Step 6: Full Audit Trail
Every write to this table is immutable and permanently auditable.
No event can be silently overwritten - only appended or versioned.


In [0]:
print("Complete Delta commit history:")
spark.sql("""
    DESCRIBE HISTORY workspace.supply_agent_demo.world_state
""").select("version", "timestamp", "operation", "userMetadata").show(truncate=False)


## Step 7: RESTORE
If needed, the corrupted state can be restored for re-processing
with corrected data quality checks applied against the raw bad values.

```sql
RESTORE TABLE workspace.supply_agent_demo.world_state TO VERSION AS OF 1
```
